# Использование KV-Cache

При тренировке модели мы обрабатывали сразу всю последовательность (или даже батчи запросов) за один прямой и обратный проход через LLM. Чтобы модель не «жульничала» и не использовала последующие токены при пересчёте скрытых представлений предыдущих, мы просто маскировали qk-скоры.

Когда LLM генерирует текст, задача меняется — так как для генерации каждого последующего токена на вход в LLM подаётся вся предыдущая последовательность. Из-за этого в наивной реализации внимания на каждом шаге повторяются вычисления одних и тех же векторов для key, query и value для всех предыдущих токенов. Такие многократные повторения приводят к квадратичному росту вычислений. 

Устранить эти повторения помогает KV-cache. В наивной реализации создают два буфера в памяти GPU: для ключей и для значений. Эти буферы постепенно заполняются по мере генерации ответа.

### Задание 1
Реализуйте наивную версию KV-кеша. 

В коде ниже — шаблон класса, реализующего вычисление внимания с использованием KV-кеша. Метод generation_step применяет функцию внимания к новым входящим токенам new_token_embeddings, используя накопленный на этом шаге KV-кеша. 

In [1]:
import torch
import torch.nn.functional as F
import numpy as np

class AttentionWithKVCache:
    def __init__(self, hidden_size=64):
        self.hidden_size = hidden_size
        self.q_proj = torch.nn.Linear(hidden_size, hidden_size)
        self.k_proj = torch.nn.Linear(hidden_size, hidden_size)
        self.v_proj = torch.nn.Linear(hidden_size, hidden_size)
        
        # Инициализация весов для воспроизводимости
        torch.manual_seed(42)
        for layer in [self.q_proj, self.k_proj, self.v_proj]:
            torch.nn.init.xavier_uniform_(layer.weight)
            torch.nn.init.zeros_(layer.bias)
    
    def generation_step(self, new_token_embeddings, cache_k, cache_v):
        """
        new_token_embeddings: [batch_size, 1, hidden_size] - эмбеддинг нового токена
        cache_k, cache_v: [batch_size, seq_len, hidden_size] - текущее состояние кеша
        """
        # Вычисляем K и V для нового токена
        new_k = self.k_proj(new_token_embeddings) # [batch_size, 1, hidden_size]
        new_v = self.v_proj(new_token_embeddings) # [batch_size, 1, hidden_size]

        # Конкатенируем новые K и V с кешем
        cache_k = torch.concat([cache_k, new_k], dim=1)
        cache_v = torch.concat([cache_v, new_v], dim=1)

        # Вычисляем Query только для нового токена
        q = self.q_proj(new_token_embeddings) # [batch_size, 1, hidden_size]

        # Используем весь кеш для вычисления внимания
        attn_weights = torch.matmul(q, cache_k.transpose(1, 2)) / (self.hidden_size ** 0.5)
        attn_weights = F.softmax(attn_weights, dim=-1)
        attn_output = torch.matmul(attn_weights, cache_v)

        return attn_output, cache_k, cache_v

In [10]:
# Тесты корректности реализации
def test_attention_with_kv_cache():
    print("Тестирование внимания с KV-кешем...")
    
    # Параметры теста
    batch_size = 2
    hidden_size = 8
    seq_len = 3
    
    # Создаём модель
    attention = AttentionWithKVCache(hidden_size)
    
    # Тест 1: Проверка согласованности с полным вниманием
    print("1. Тест согласованности с полным вниманием:")
    
    # Создаём полную последовательность
    full_sequence = torch.randn(batch_size, seq_len, hidden_size)
    
    # Вычисляем K, V, Q для всей последовательности
    K_full = attention.k_proj(full_sequence)
    V_full = attention.v_proj(full_sequence)
    Q_full = attention.q_proj(full_sequence[:, -1:, :])  # Query только для последнего токена
    
    # Вычисляем полное внимание
    attn_weights_full = torch.matmul(Q_full, K_full.transpose(1, 2)) / (hidden_size ** 0.5)
    attn_weights_full = F.softmax(attn_weights_full, dim=-1)
    expected_output = torch.matmul(attn_weights_full, V_full)
    
    # Вычисляем пошагово с кешем
    cache_k = torch.zeros((batch_size, 0, hidden_size))
    cache_v = torch.zeros((batch_size, 0, hidden_size))
    
    for i in range(seq_len):
        token_emb = full_sequence[:, i:i+1, :]
        output, cache_k, cache_v = attention.generation_step(token_emb, cache_k, cache_v)
    
    # Сравниваем результаты
    assert torch.allclose(output, expected_output, atol=1e-6), "Результаты не совпадают с полным вниманием"
    assert torch.allclose(cache_k, K_full, atol=1e-6), "Кеш K не совпадает"
    assert torch.allclose(cache_v, V_full, atol=1e-6), "Кеш V не совпадает"
    
    print("Согласованность с полным вниманием подтверждена")
    
    # Тест 2: Проверка на граничных случаях
    print("2. Тест граничных случаев:")
    
    # Нулевой вход
    zero_input = torch.zeros(1, 1, hidden_size)
    cache_k_zero = torch.zeros(1, 0, hidden_size)
    cache_v_zero = torch.zeros(1, 0, hidden_size)
    
    output_zero, _, _ = attention.generation_step(zero_input, cache_k_zero, cache_v_zero)
    
    # Проверяем, что выход не NaN
    assert not torch.isnan(output_zero).any(), "Обнаружены NaN-значения"
    
    print("Граничные случаи обрабатываются корректно")
    
    print("Все тесты пройдены успешно!")

# Пример использования
def example_usage():
    print("Пример использования внимания с KV-кешем:")
    print("-" * 50)
    
    # Инициализация
    hidden_size = 64
    batch_size = 1
    attention = AttentionWithKVCache(hidden_size)
    
    # Инициализируем кеши как пустые тензоры
    cache_k = torch.zeros((batch_size, 0, hidden_size))
    cache_v = torch.zeros((batch_size, 0, hidden_size))
    
    # Генерация нескольких токенов
    num_tokens = 5
    print(f"Генерируем {num_tokens} токенов:")
    
    for step in range(num_tokens):
        # Создаём случайный эмбеддинг нового токена (в реальности это был бы эмбеддинг сгенерированного токена)
        new_token_emb = torch.randn(batch_size, 1, hidden_size)
        
        # Выполняем шаг генерации
        attn_output, cache_k, cache_v = attention.generation_step(new_token_emb, cache_k, cache_v)
        
        print(f"Шаг {step + 1}:")
        print(f"  Размер кеша K: {cache_k.shape}")
        print(f"  Размер кеша V: {cache_v.shape}")
        print(f"  Выход внимания: {attn_output.shape}")
    
    print("\nГенерация завершена!")

# Запуск тестов и примера
if __name__ == "__main__":
    # Запускаем тесты
    test_attention_with_kv_cache()
    print("\n")
    
    # Запускаем пример использования
    example_usage()


Тестирование внимания с KV-кешем...
1. Тест согласованности с полным вниманием:
Согласованность с полным вниманием подтверждена
2. Тест граничных случаев:
Граничные случаи обрабатываются корректно
Все тесты пройдены успешно!


Пример использования внимания с KV-кешем:
--------------------------------------------------
Генерируем 5 токенов:
Шаг 1:
  Размер кеша K: torch.Size([1, 1, 64])
  Размер кеша V: torch.Size([1, 1, 64])
  Выход внимания: torch.Size([1, 1, 64])
Шаг 2:
  Размер кеша K: torch.Size([1, 2, 64])
  Размер кеша V: torch.Size([1, 2, 64])
  Выход внимания: torch.Size([1, 1, 64])
Шаг 3:
  Размер кеша K: torch.Size([1, 3, 64])
  Размер кеша V: torch.Size([1, 3, 64])
  Выход внимания: torch.Size([1, 1, 64])
Шаг 4:
  Размер кеша K: torch.Size([1, 4, 64])
  Размер кеша V: torch.Size([1, 4, 64])
  Выход внимания: torch.Size([1, 1, 64])
Шаг 5:
  Размер кеша K: torch.Size([1, 5, 64])
  Размер кеша V: torch.Size([1, 5, 64])
  Выход внимания: torch.Size([1, 1, 64])

Генерация заверше

## Packing для квантизации менее 4 бит
Выполняя элементарные операции над числами (сложение, умножение и т. д.), GPU использует разные инструкции, рассчитанные под определённую точность. Так, для вычислений во float32 и int8 используются одинаковые процессоры, но с разными инструкциями. При этом набор инструкций у каждой модели GPU ограничен. Например, часто используемая для инференса A100 поддерживает вычисления только в FP64, TF32, FP32, FP16, BF16, INT8.

Техники квантизации, которые мы рассмотрели, сжимают веса до разной точности. Например, методы AWQ и GPTQ сохраняют качество как при минимальной квантизации в 8 бит, так и при экстремальной квантизации менее 4 бит. В то же время часто используемая A100 не поддерживает вычисления в 4 bit формате нативно. 

Решение заключается в том, что несколько низкобитных значений упаковывают в один стандартный контейнер. 

Рассмотрим пример упаковки 3-битных весов. Хранить два значения в одном байте (6 бит из 8) неэффективно, лучше упакуем на уровне нескольких байтов. Например, упакуем 8 значений по 3 бита: 

8 значений × 3 бита/значение = 24 бита = 3 байт.

В VLLM и других фреймворках такие операции реализованы в высокооптимизированных CUDA-ядрах, которые эффективно распаковывают веса прямо перед матричным умножением. Для образовательных целей предлагаем реализовать packing на простой задаче. Так вы лучше поймёте, как работает подход и в чём его особенности. 

### Задание 2
Перед вами шаблон кода для упаковки и распаковки чисел в 3-битный формат. 

На входе есть массив чисел, каждое из которых можно представить в виде 3 битов в двоичной форме. 

Допишите реализацию описанных методов. Нужно сжать (упаковать) массив из 8 чисел в 3 более крупные числа, представимые в виде 8 бит информации в двоичном виде.

Для проверки корректности добавлены логи, они помогут сравнить совпадение массива до применения методов и после цикла запаковка/распаковка. 

In [2]:
values_3bit = [1, 5, 0, 7, 2, 3, 4, 6]
packed_bytes = bytearray(3)

# Упаковка
for i, val in enumerate(values_3bit):
    byte_index = (i * 3) // 8
    bit_offset = (i * 3) % 8
    
    # Ограничиваем значение делением на 8 (остаток от деления)
    val = val % 8  # эквивалентно val & 0x07
    
    # Записываем значение
    shifted = val << bit_offset
    packed_bytes[byte_index] |= shifted % 256  # берём остаток от 256
    
    if bit_offset > 5:
        remaining = val >> (8 - bit_offset)
        packed_bytes[byte_index + 1] |= remaining % 256

print(f"Исходные значения: {values_3bit}")
print(f"Упакованные байты: {[b for b in packed_bytes]}")

# Распаковка
unpacked_values = []
for i in range(8):
    byte_index = (i * 3) // 8
    bit_offset = (i * 3) % 8
    
    # Извлекаем значение из текущего байта
    value = (packed_bytes[byte_index] >> bit_offset) % 8
    
    # Если значение пересекает границу байта, добавляем биты из следующего байта
    if bit_offset > 5:
        bits_from_next = 8 - bit_offset
        next_part = (packed_bytes[byte_index + 1] << bits_from_next) % 256
        value = (value + next_part) % 8
    
    unpacked_values.append(value)

print(f"Распакованные значения: {unpacked_values}")
print(f"Совпадение: {values_3bit == unpacked_values}")

Исходные значения: [1, 5, 0, 7, 2, 3, 4, 6]
Упакованные байты: [41, 174, 209]
Распакованные значения: [1, 5, 0, 7, 2, 3, 4, 6]
Совпадение: True
